# Rebuttal Results

Focused notebook for rebuttal-only results: LoRA and dense full-finetune scale sweeps, adaptive/action-space/up-cost baselines, and training efficiency/stability tables and reward/cost plots for full-finetune vs LoRA.

## Re-run Commands

Adaptive rebuttal sweep:

```bash
BLEND_GPUS=0,1,2,3 ADAPTIVE_MODEL="trained_models/LoraF_invi_visi_rank_1" CHECKPOINT="03400.pt" EXP_NOTE="timetest" SEEDS="42 1000 2000 3000 4000" ADAPTIVE_BEHAVIOURS="adaptive_gt adaptive_action_gt adaptive_fullfinetune_gt Gensafenav_cons_upcost mpc_adaptive" TEST_SIZE=250 HUMAN_NUM=20 AWARENESS_EVAL=off MAX_PARALLEL=4 ./test_adaptive_lora_poc.sh
```

LoRA scale sweep:

```bash
for SEED in 42 1000 2000 3000 4000; do
  SEED="$SEED" EXP_ID="$SEED" ./test_lora_ablation.sh
done
```


Action-space interpolation scale sweep:

```bash
./test_action_space_ablation.sh
```

MPC fixed-kappa clearance sweep:

```bash
./test_mpc_ablation.sh
```

Dense full-finetune interpolation sweep:

```bash
./test_fullfinetune_ablation.sh
```

Training timing comparison configs should use the same backbone and same training flags for `Fullfinetune_timetest`, `Lora1_timetest_rank_1`, and `Lora4_timetest_rank_4`.

In [ ]:
# ----------------------- setup -----------------------
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

SCENARIOS = [
    'seperate_all_ignorant',
    'seperate_all_aware',
    'seperate_mixed_5050',
    'cluster_aware_ignorant',
]

SCENARIO_LABELS = {
    'seperate_all_ignorant': 'Non-aware',
    'seperate_all_aware': 'Aware',
    'seperate_mixed_5050': 'Mixed',
    'cluster_aware_ignorant': 'Spatial Clusters',
}
SCENARIO_DISPLAY_ORDER = SCENARIOS
SEEDS = ['42', '1000', '2000', '3000', '4000']

METRICS = [
    ('success_rate', 'SR (%)', True, 2),
    ('collision_rate', 'CR (%)', True, 2),
    ('avg_nav_time', 'NT (s)', False, 2),
    ('avg_path_length', 'PL (m)', False, 2),
    ('avg_intrusion_ratio_pct', 'ITR (%)', False, 2),
    ('avg_min_social_distance', 'SocD (m)', False, 3),
    ('avg_lora_scale', 'Avg scale', False, 3),
    ('avg_inference_time_ms', 'Inf (ms)', False, 3),
    ('p95_inference_time_ms', 'P95 Inf (ms)', False, 3),
    ('max_inference_peak_gpu_memory_mb', 'Peak GPU (MB)', False, 1),
    ('avg_matrix_calc_time_ms', 'Mat (ms)', False, 3),
    ('p95_matrix_calc_time_ms', 'P95 Mat (ms)', False, 3),
    ('num_episodes', 'N', False, 0),
]


def fmt_stat(value, decimals):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return '—'
    return f'{value:.{decimals}f}'


def metric_stat_columns(metrics):
    cols = []
    for _, label, _, _ in metrics:
        if label == 'N':
            cols.append((label, 'total'))
        else:
            cols.extend([(label, 'mean'), (label, 'std')])
    return cols


def add_metric_stats(row, metric_key, label, in_pct, decimals, vals):
    vals = [v for v in vals if v is not None]
    if not vals:
        if label == 'N':
            row[f'{label} total'] = '—'
        else:
            row[f'{label} mean'] = '—'
            row[f'{label} std'] = '—'
        return

    if metric_key == 'num_episodes':
        row[f'{label} total'] = str(int(np.sum(vals)))
        return

    mean = float(np.mean(vals))
    sd = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
    if in_pct:
        mean *= 100
        sd *= 100
    row[f'{label} mean'] = fmt_stat(mean, decimals)
    row[f'{label} std'] = fmt_stat(sd, decimals)


def load_aggregate(model_dir):
    path = Path(model_dir) / 'test' / 'all_evaluations.json'
    if not path.exists():
        return {}, str(path)
    with path.open() as f:
        return json.load(f), str(path)


def load_method(model_dir, behaviour, seed_set):
    data, _ = load_aggregate(model_dir)
    out = {sc: {} for sc in SCENARIOS}
    for key, value in data.items():
        if '_exp' not in key:
            continue
        base, exp = key.rsplit('_exp', 1)
        if exp not in seed_set:
            continue
        if not base.endswith('_' + behaviour):
            continue
        scenario = base[:-(len(behaviour) + 1)]
        if scenario not in SCENARIOS:
            continue
        out[scenario][exp] = value.get('summary', {})
    return out


def build_scenario_first_table(loaded, methods, metrics):
    rows = []
    for label, _, _ in methods:
        for scenario in SCENARIOS:
            row = {'Method': label, 'Scenario': scenario}
            seed_dicts = loaded[label][scenario]
            for mkey, mlabel, in_pct, decimals in metrics:
                vals = [summary.get(mkey) for summary in seed_dicts.values() if summary.get(mkey) is not None]
                add_metric_stats(row, mkey, mlabel, in_pct, decimals, vals)
            rows.append(row)

    df_long = pd.DataFrame(rows)
    value_cols = [c for c in df_long.columns if c not in ('Method', 'Scenario')]
    melted = df_long.melt(id_vars=['Method', 'Scenario'], value_vars=value_cols, var_name='MetricStat', value_name='Value')
    melted[['Metric', 'Stat']] = melted['MetricStat'].str.rsplit(' ', n=1, expand=True)
    melted['Scenario group'] = melted['Scenario'].map(SCENARIO_LABELS).fillna(melted['Scenario'])
    wide = melted.pivot_table(index=['Scenario group', 'Method'], columns=['Metric', 'Stat'], values='Value', aggfunc='first')
    wide = wide.reindex(pd.MultiIndex.from_tuples(
        [(SCENARIO_LABELS[sc], label) for sc in SCENARIO_DISPLAY_ORDER for label, _, _ in methods],
        names=['Scenario', 'Method'],
    ))
    wide = wide.reindex(columns=pd.MultiIndex.from_tuples(metric_stat_columns(metrics), names=['Metric', 'Stat']))
    return df_long, wide


def coverage_table(loaded, methods):
    rows = []
    for label, _, seeds in methods:
        for scenario in SCENARIOS:
            found = sorted(loaded[label][scenario].keys())
            rows.append({
                'Method': label,
                'Scenario': SCENARIO_LABELS.get(scenario, scenario),
                'Expected seeds': len(seeds),
                'Found seeds': len(found),
                'Complete': set(seeds).issubset(found),
                'Seeds': found,
            })
    return pd.DataFrame(rows)


## LoRA Scale Sweep

Rows are fixed action-network LoRA scales plus the existing dynamic `adaptive_gt`. Tables are scenario-first and report mean/std across seeds.

In [ ]:
# ----------------------- LoRA scale sweep -----------------------
LORA_SWEEP_DIR = 'trained_models/LoraF_invi_visi_rank_1'
LORA_SWEEP_SEEDS = SEEDS


def scale_seeds(scale_str):
    return {f'scale_{scale_str}_{seed}' for seed in LORA_SWEEP_SEEDS}

LORA_SWEEP_METHODS = [
    ('LoRA scale 0.0', 'always_off', set(LORA_SWEEP_SEEDS)),
    ('LoRA scale 0.2', 'fixed_scale', scale_seeds('0.2')),
    ('LoRA scale 0.4', 'fixed_scale', scale_seeds('0.4')),
    ('LoRA scale 0.6', 'fixed_scale', scale_seeds('0.6')),
    ('LoRA scale 0.8', 'fixed_scale', scale_seeds('0.8')),
    ('LoRA scale 1.0', 'fixed_scale', scale_seeds('1.0')),
    ('Adaptive GT', 'adaptive_gt', set(LORA_SWEEP_SEEDS)),
]

lora_loaded = {label: load_method(LORA_SWEEP_DIR, behaviour, seeds) for label, behaviour, seeds in LORA_SWEEP_METHODS}
display(coverage_table(lora_loaded, LORA_SWEEP_METHODS))
_, lora_sweep_table = build_scenario_first_table(lora_loaded, LORA_SWEEP_METHODS, METRICS)
lora_sweep_table


## Action-Space Interpolation Scale Sweep

Fixed action-space interpolation sweep from conservative endpoint actions to cooperative endpoint actions. Unlike LoRA scale sweep, interpolation happens after both endpoint actions are computed.

In [ ]:
# ----------------------- action-space interpolation scale sweep -----------------------
ACTION_SWEEP_DIR = 'trained_models/LoraF_invi_visi_rank_1'
ACTION_SWEEP_SEEDS = SEEDS


def action_scale_seeds(scale_str):
    return {f'action_scale_{scale_str}_{seed}' for seed in ACTION_SWEEP_SEEDS}

ACTION_SWEEP_METHODS = [
    ('action scale 0.0', 'fixed_action_scale', action_scale_seeds('0.0')),
    ('action scale 0.2', 'fixed_action_scale', action_scale_seeds('0.2')),
    ('action scale 0.4', 'fixed_action_scale', action_scale_seeds('0.4')),
    ('action scale 0.6', 'fixed_action_scale', action_scale_seeds('0.6')),
    ('action scale 0.8', 'fixed_action_scale', action_scale_seeds('0.8')),
    ('action scale 1.0', 'fixed_action_scale', action_scale_seeds('1.0')),
]

action_loaded = {label: load_method(ACTION_SWEEP_DIR, behaviour, seeds) for label, behaviour, seeds in ACTION_SWEEP_METHODS}
display(coverage_table(action_loaded, ACTION_SWEEP_METHODS))
_, action_sweep_table = build_scenario_first_table(action_loaded, ACTION_SWEEP_METHODS, METRICS)
action_sweep_table


## MPC Fixed-kappa Clearance Sweep

Fixed `k_t` sweep for the MPC baseline. Clearance is computed as `d_min = 1 - k_t`, with the horizon matched to the GST future prediction length.

In [ ]:
# ----------------------- MPC fixed-kappa clearance sweep -----------------------
MPC_SWEEP_DIR = 'trained_models/LoraF_invi_visi_rank_1'
MPC_SWEEP_SEEDS = SEEDS


def mpc_scale_seeds(scale_str):
    return {f'mpc_scale_{scale_str}_{seed}' for seed in MPC_SWEEP_SEEDS}


MPC_METRICS = METRICS + [
    ('avg_mpc_solve_time_ms', 'MPC Solve (ms)', False, 3),
    ('p95_mpc_solve_time_ms', 'P95 MPC (ms)', False, 3),
    ('avg_mpc_dmin', 'Avg d_min', False, 3),
    ('avg_mpc_sensed_humans', 'Sensed H', False, 2),
    ('avg_mpc_min_human_distance', 'MPC Min Dist', False, 3),
]

MPC_SWEEP_METHODS = [
    ('MPC k_t 0.0', 'mpc_fixed', mpc_scale_seeds('0.0')),
    ('MPC k_t 0.2', 'mpc_fixed', mpc_scale_seeds('0.2')),
    ('MPC k_t 0.4', 'mpc_fixed', mpc_scale_seeds('0.4')),
    ('MPC k_t 0.6', 'mpc_fixed', mpc_scale_seeds('0.6')),
    ('MPC k_t 0.8', 'mpc_fixed', mpc_scale_seeds('0.8')),
    ('MPC k_t 1.0', 'mpc_fixed', mpc_scale_seeds('1.0')),
]

mpc_loaded = {label: load_method(MPC_SWEEP_DIR, behaviour, seeds) for label, behaviour, seeds in MPC_SWEEP_METHODS}
display(coverage_table(mpc_loaded, MPC_SWEEP_METHODS))
_, mpc_sweep_table = build_scenario_first_table(mpc_loaded, MPC_SWEEP_METHODS, MPC_METRICS)
mpc_sweep_table


## Full-Finetune Scale Sweep

Dense interpolation sweep from the conservative backbone to the full-finetuned endpoint, including the adaptive dense interpolation baseline.

In [ ]:
# ----------------------- dense full-finetune scale sweep -----------------------
FULLFT_SWEEP_DIR = 'trained_models/LoraF_invi_visi_rank_1'
FULLFT_SWEEP_SEEDS = SEEDS


def fullft_scale_seeds(scale_str):
    return {f'fullft_scale_{scale_str}_{seed}' for seed in FULLFT_SWEEP_SEEDS}

FULLFT_SWEEP_METHODS = [
    ('fullft scale 0.0', 'fixed_fullfinetune_scale', fullft_scale_seeds('0.0')),
    ('fullft scale 0.2', 'fixed_fullfinetune_scale', fullft_scale_seeds('0.2')),
    ('fullft scale 0.4', 'fixed_fullfinetune_scale', fullft_scale_seeds('0.4')),
    ('fullft scale 0.6', 'fixed_fullfinetune_scale', fullft_scale_seeds('0.6')),
    ('fullft scale 0.8', 'fixed_fullfinetune_scale', fullft_scale_seeds('0.8')),
    ('fullft scale 1.0', 'fixed_fullfinetune_scale', fullft_scale_seeds('1.0')),
    ('Adaptive full-finetune GT', 'adaptive_fullfinetune_gt', set(FULLFT_SWEEP_SEEDS)),
]

fullft_loaded = {label: load_method(FULLFT_SWEEP_DIR, behaviour, seeds) for label, behaviour, seeds in FULLFT_SWEEP_METHODS}
display(coverage_table(fullft_loaded, FULLFT_SWEEP_METHODS))
_, fullft_sweep_table = build_scenario_first_table(fullft_loaded, FULLFT_SWEEP_METHODS, METRICS)
fullft_sweep_table


## Adaptive Rebuttal Baselines

Compares the existing weight-space adaptive GT method against action-space interpolation, dense full-finetune interpolation, and the conservative up-cost GenSafeNav baseline. Set `REBUTTAL_EXP_NOTE` to the note used in `test_adaptive_lora_poc.sh`.

In [ ]:
# ----------------------- adaptive rebuttal performance -----------------------
REBUTTAL_EXP_NOTE = 'timetest'
REBUTTAL_EXP_ID_TEMPLATE = '{seed}'
REBUTTAL_SEEDS = SEEDS
REBUTTAL_METHODS = [
    ('Adaptive GT', 'trained_models/LoraF_invi_visi_rank_1', 'adaptive_gt'),
    ('Action-space adaptive GT', 'trained_models/LoraF_invi_visi_rank_1', 'adaptive_action_gt'),
    ('Adaptive full-finetune GT', 'trained_models/LoraF_invi_visi_rank_1', 'adaptive_fullfinetune_gt'),
    ('Gensafenav_cons_upcost', 'trained_models/Conservative_Backbone_CostLimit_1.2', 'Gensafenav_cons_upcost'),
    ('Adaptive MPC', 'trained_models/LoraF_invi_visi_rank_1', 'mpc_adaptive'),
]

coverage_rows = []
table_rows = []
for method_name, model_dir, behaviour in REBUTTAL_METHODS:
    aggregate, agg_path = load_aggregate(model_dir)
    for scenario in SCENARIOS:
        summaries = {}
        for seed in REBUTTAL_SEEDS:
            exp_id = f'{REBUTTAL_EXP_NOTE}_{seed}' if REBUTTAL_EXP_NOTE else REBUTTAL_EXP_ID_TEMPLATE.format(seed=seed)
            key = f'{scenario}_{behaviour}_exp{exp_id}'
            ent = aggregate.get(key)
            if ent is not None:
                summaries[seed] = ent.get('summary', {})

        found = sorted(summaries.keys(), key=lambda s: REBUTTAL_SEEDS.index(s))
        coverage_rows.append({
            'Method': method_name,
            'Behaviour': behaviour,
            'Model dir': model_dir,
            'Aggregate': agg_path,
            'Scenario': SCENARIO_LABELS.get(scenario, scenario),
            'Exp note': REBUTTAL_EXP_NOTE,
            'Expected seeds': len(REBUTTAL_SEEDS),
            'Found seeds': len(found),
            'Complete': len(found) == len(REBUTTAL_SEEDS),
            'Seeds': found,
        })

        row = {'Method': method_name, 'Scenario': scenario}
        for mkey, mlabel, in_pct, decimals in METRICS:
            vals = [summary.get(mkey) for summary in summaries.values() if summary.get(mkey) is not None]
            add_metric_stats(row, mkey, mlabel, in_pct, decimals, vals)
        table_rows.append(row)

rebuttal_coverage = pd.DataFrame(coverage_rows)
display(rebuttal_coverage)

rebuttal_df = pd.DataFrame(table_rows)
value_cols = [c for c in rebuttal_df.columns if c not in ('Method', 'Scenario')]
rebuttal_long = rebuttal_df.melt(id_vars=['Method', 'Scenario'], value_vars=value_cols, var_name='MetricStat', value_name='Value')
rebuttal_long[['Metric', 'Stat']] = rebuttal_long['MetricStat'].str.rsplit(' ', n=1, expand=True)
rebuttal_long['Scenario group'] = rebuttal_long['Scenario'].map(SCENARIO_LABELS).fillna(rebuttal_long['Scenario'])
rebuttal_table = rebuttal_long.pivot_table(index=['Scenario group', 'Method'], columns=['Metric', 'Stat'], values='Value', aggfunc='first')
rebuttal_table = rebuttal_table.reindex(pd.MultiIndex.from_tuples(
    [(SCENARIO_LABELS[sc], name) for sc in SCENARIOS for name, _, _ in REBUTTAL_METHODS],
    names=['Scenario', 'Method'],
))
rebuttal_table = rebuttal_table.reindex(columns=pd.MultiIndex.from_tuples(metric_stat_columns(METRICS), names=['Metric', 'Stat']))
rebuttal_table


In [ ]:
# ----------------------- fixed old inference timing data -----------------------
# Uses the original rebuttal timing table values provided for the plot.
inference_old_data = pd.DataFrame([
    {
        'Method': 'Adaptive GT',
        'Inference time (ms) mean': 1.4155,
        'Inference time (ms) std': 0.02955785288,
        'P95 Inf (ms) mean': 1.51875,
        'P95 Inf (ms) std': 0.05081584399,
        'Peak GPU (MB) mean': 41.5,
        'Peak GPU (MB) std': 0.0,
    },
    {
        'Method': 'Action-space adaptive GT',
        'Inference time (ms) mean': 2.75225,
        'Inference time (ms) std': 0.03682729966,
        'P95 Inf (ms) mean': 2.93325,
        'P95 Inf (ms) std': 0.08155723961,
        'Peak GPU (MB) mean': 41.5,
        'Peak GPU (MB) std': 0.0,
    },
    {
        'Method': 'Adaptive full-finetune GT',
        'Inference time (ms) mean': 1.89675,
        'Inference time (ms) std': 0.01866145761,
        'P95 Inf (ms) mean': 2.25725,
        'P95 Inf (ms) std': 0.02328626204,
        'Peak GPU (MB) mean': 60.6,
        'Peak GPU (MB) std': 0.0,
    },
    {
        'Method': 'Gensafenav_cons_upcost',
        'Inference time (ms) mean': 1.18325,
        'Inference time (ms) std': 0.02139119757,
        'P95 Inf (ms) mean': 1.26025,
        'P95 Inf (ms) std': 0.01412739655,
        'Peak GPU (MB) mean': 36.0,
        'Peak GPU (MB) std': 0.0,
    },
]).set_index('Method')

display(inference_old_data)


## SR-PL Rebuttal Sweep Plot

Success-rate versus path-length tradeoff plots for Ours, dense full-finetune, action-space interpolation, and MPC rebuttal sweeps.

In [ ]:
# ----------------------- SR-PL rebuttal sweep plots -----------------------
from plot_sr_pl_tradeoff import (
    OURS_DATA,
    load_latest_fullft_data,
    load_latest_action_data,
    load_latest_mpc_data,
    plot_tradeoffs,
)

SRPL_AGGREGATE = 'trained_models/LoraF_invi_visi_rank_1/test/all_evaluations.json'
srpl_fullft = load_latest_fullft_data(SRPL_AGGREGATE)
srpl_action = load_latest_action_data(SRPL_AGGREGATE)
srpl_mpc = load_latest_mpc_data(SRPL_AGGREGATE)

for name, rows in [
    ('full-finetune', srpl_fullft),
    ('action-space', srpl_action),
    ('MPC', srpl_mpc),
]:
    if not rows:
        print(f'Warning: no {name} SR-PL sweep rows found in {SRPL_AGGREGATE}')

srpl_df = pd.DataFrame(
    OURS_DATA + srpl_fullft + srpl_action + srpl_mpc,
    columns=['Method', 'Scenario', 'Scale', 'SR', 'CR', 'NT', 'PL'],
)

srpl_saved = plot_tradeoffs(srpl_df, '/tmp/sr_pl_rebuttal_sweeps', show=True)
srpl_df


## Training Comparison

Training table and reward/cost curves for dense full-finetune versus LoRA rank 1 and rank 4. `train_time` is optimizer/update time accumulated by `train.py`; `sim_time` is rollout/environment collection time.

In [ ]:
# ----------------------- training summaries -----------------------
TRAINING_RUNS = [
    ('LoRA rank 1', 'trained_models/Lora1_timetest_rank_1'),
    ('LoRA rank 4', 'trained_models/Lora4_timetest_rank_4'),
    ('Dense full-finetune delta', 'trained_models/Fullfinetune_timetest'),
]


def load_progress(model_dir):
    path = Path(model_dir) / 'progress.csv'
    if not path.exists():
        return None
    df = pd.read_csv(path)
    numeric_cols = [c for c in df.columns if c != 'misc/nupdates']
    for col in numeric_cols + ['misc/nupdates']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

progress = {}
for label, model_dir in TRAINING_RUNS:
    df = load_progress(model_dir)
    if df is not None:
        progress[label] = df.assign(Method=label, Model=model_dir)

summary_rows = []
for label, model_dir in TRAINING_RUNS:
    df = progress.get(label)
    if df is None or df.empty:
        summary_rows.append({'Method': label, 'Model dir': model_dir, 'Status': 'missing progress.csv'})
        continue

    final = df.iloc[-1]
    best_reward = df.loc[df['eprewmean'].idxmax()]
    best_sr = df.loc[df['epsuccessmean'].idxmax()]
    min_cost = df.loc[df['epcostmean'].idxmin()]
    summary_rows.append({
        'Method': label,
        'Model dir': model_dir,
        'Rows': len(df),
        'Final update': int(final['misc/nupdates']),
        'Final env steps': int(final['misc/total_timesteps']),
        'Final reward': round(float(final['eprewmean']), 3),
        'Final cost': round(float(final['epcostmean']), 4),
        'Final SR (%)': round(float(final['epsuccessmean']) * 100, 2),
        'Best reward': round(float(best_reward['eprewmean']), 3),
        'Best reward update': int(best_reward['misc/nupdates']),
        'Best SR (%)': round(float(best_sr['epsuccessmean']) * 100, 2),
        'Best SR update': int(best_sr['misc/nupdates']),
        'Min cost': round(float(min_cost['epcostmean']), 4),
        'Min cost update': int(min_cost['misc/nupdates']),
        'Update train time (h)': round(float(final['train_time']) / 3600, 3),
        'Sim time (h)': round(float(final['sim_time']) / 3600, 3),
        'Mean update time (s)': round(float(df['update_time_per_iter'].mean()), 4),
        'P95 update time (s)': round(float(df['update_time_per_iter'].quantile(0.95)), 4),
        'Max peak GPU (MB)': round(float(df['peak_gpu_memory_mb'].max()), 1),
    })

training_summary = pd.DataFrame(summary_rows)
training_summary


In [ ]:
# ----------------------- training and inference cost bars -----------------------
# Left: optimizer/update GPU-hours from train.py train_time. Right: old inference timing values.
training_bar_rows = []
for label, model_dir in TRAINING_RUNS:
    df = progress.get(label)
    if df is None or df.empty:
        continue
    final = df.iloc[-1]
    training_bar_rows.append({
        'Method': label,
        'Update GPU-hours': float(final['train_time']) / 3600.0,
    })

training_bar_df = pd.DataFrame(training_bar_rows).set_index('Method').reindex([label for label, _ in TRAINING_RUNS])
display(training_bar_df.round({'Update GPU-hours': 3}))

TITLE_SIZE = 18
LABEL_SIZE = 15
TICK_SIZE = 12
VALUE_SIZE = 12
LEGEND_SIZE = 13

train_x = np.arange(len(training_bar_df.index))
inf_methods = inference_old_data.index.tolist()
inf_x = np.arange(len(inf_methods))

train_color = '#D62728'
inf_color = '#D62728'
edge_color = '#202020'

fig, axes = plt.subplots(1, 2, figsize=(12, 6), constrained_layout=True)
ax_train, ax_inf = axes

train_vals = training_bar_df['Update GPU-hours'].values
train_bars = ax_train.bar(
    train_x, train_vals,
    color=train_color, edgecolor=edge_color, linewidth=0.8,
    alpha=0.92, zorder=3,
)
ax_train.set_title('Training Update Cost', fontsize=TITLE_SIZE, fontweight='bold', pad=14)
ax_train.set_ylabel('Update GPU-hours', fontsize=LABEL_SIZE, fontweight='bold')
ax_train.set_xticks(train_x)
ax_train.set_xticklabels(training_bar_df.index, rotation=18, ha='right', fontsize=TICK_SIZE)
ax_train.tick_params(axis='y', labelsize=TICK_SIZE)
ax_train.grid(axis='y', alpha=0.25, linestyle='--', zorder=0)
ax_train.set_axisbelow(True)
ax_train.set_box_aspect(1)
ax_train.set_ylim(0, max(float(np.nanmax(train_vals)) * 1.22, 1.0))

for bar, value in zip(train_bars, train_vals):
    if pd.isna(value):
        continue
    ax_train.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + ax_train.get_ylim()[1] * 0.025,
        f'{value:.2f}',
        ha='center', va='bottom', fontsize=VALUE_SIZE, fontweight='bold', color=train_color,
    )

inf_means = inference_old_data['Inference time (ms) mean']
inf_stds = inference_old_data['Inference time (ms) std']
inf_bars = ax_inf.bar(
    inf_x, inf_means.values,
    yerr=inf_stds.values, capsize=5,
    color=inf_color, edgecolor=edge_color, linewidth=0.8,
    alpha=0.92, zorder=3,
)
ax_inf.set_title('Inference Time', fontsize=TITLE_SIZE, fontweight='bold', pad=14)
ax_inf.set_ylabel('Inference time (ms)', fontsize=LABEL_SIZE, fontweight='bold')
ax_inf.set_xticks(inf_x)
ax_inf.set_xticklabels(inf_methods, rotation=18, ha='right', fontsize=TICK_SIZE)
ax_inf.tick_params(axis='y', labelsize=TICK_SIZE)
ax_inf.grid(axis='y', alpha=0.25, linestyle='--', zorder=0)
ax_inf.set_axisbelow(True)
ax_inf.set_box_aspect(1)
ax_inf.set_ylim(0, max(float((inf_means + inf_stds).max()) * 1.22, 1.0))

for bar, value in zip(inf_bars, inf_means.values):
    ax_inf.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + ax_inf.get_ylim()[1] * 0.025,
        f'{value:.2f}',
        ha='center', va='bottom', fontsize=VALUE_SIZE, fontweight='bold', color=inf_color,
    )

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('Training and Inference Efficiency', fontsize=20, fontweight='bold')
plt.show()


In [ ]:
# ----------------------- reward and cost plots -----------------------
if not progress:
    print('No progress.csv files found for TRAINING_RUNS.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharex=True)
    for label, df in progress.items():
        wall_hours = (df['sim_time'] + df['train_time']) / 3600.0
        reward = df['eprewmean'].rolling(window=25, min_periods=1).mean()
        cost = df['epcostmean'].rolling(window=25, min_periods=1).mean()
        axes[0].plot(wall_hours, reward, label=label)
        axes[1].plot(wall_hours, cost, label=label)

    axes[0].set_title('Training Reward')
    axes[0].set_xlabel('Wall-clock time (h)')
    axes[0].set_ylabel('Mean episode reward')
    axes[0].grid(True, alpha=0.3)

    axes[1].set_title('Training Cost')
    axes[1].set_xlabel('Wall-clock time (h)')
    axes[1].set_ylabel('Mean episode cost')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(loc='best')

    plt.tight_layout()
    plt.show()


In [ ]:
# ----------------------- success, update time, and memory plots -----------------------
if progress:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), sharex=False)
    for label, df in progress.items():
        steps_m = df['misc/total_timesteps'] / 1e6
        sr = df['epsuccessmean'].rolling(window=25, min_periods=1).mean() * 100
        upd = df['update_time_per_iter'].rolling(window=25, min_periods=1).mean()
        mem = df['peak_gpu_memory_mb'].rolling(window=25, min_periods=1).mean()
        axes[0].plot(steps_m, sr, label=label)
        axes[1].plot(steps_m, upd, label=label)
        axes[2].plot(steps_m, mem, label=label)

    axes[0].set_title('Training Success Rate')
    axes[0].set_xlabel('Environment steps (M)')
    axes[0].set_ylabel('SR (%)')
    axes[1].set_title('PPO Update Time')
    axes[1].set_xlabel('Environment steps (M)')
    axes[1].set_ylabel('Seconds / update')
    axes[2].set_title('Peak GPU Memory During Update')
    axes[2].set_xlabel('Environment steps (M)')
    axes[2].set_ylabel('MB')
    for ax in axes:
        ax.grid(True, alpha=0.3)
    axes[2].legend(loc='best')
    plt.tight_layout()
    plt.show()
